# Native AHR improvement — shared definitions

Purpose: load explicitly exported native AHR/rules/checkpoint definitions and identify the new experiment sources. Inputs: frozen original notebooks and new AHR-only notebooks. Outputs: verified namespace and source hashes. Expected runtime: under 15 seconds. No training occurs here.

All new training uses GPU 1. Historical JEPA records stay archived. New inference and training contain no JEPA objective or predictor. Checkpoints belong in `checkpoints/<run_id>/`; exact resume commands are documented in the training and controller notebooks. The workspace is on the persistent boot disk; off-instance backup awaits the user's storage destination.


In [ ]:
from pathlib import Path
import copy, hashlib, json, os, time, uuid, random, sys
import nbformat

V4_ROOT = 'experiments/ahr_native_v4'
V4_SCHEMA = 'ahr_native_go_v4'
V4_DEVICE = 'cuda:1'
V4_NOTEBOOKS = ('00_shared.ipynb', '01_native_ahr_and_tactics.ipynb', '02_competitive_training.ipynb', '03_gnugo_evaluation.ipynb', '04_pipeline.ipynb')

def v4_project(start=None):
    project = Path.cwd().resolve() if start is None else Path(start).resolve()
    while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
        if project.parent == project: raise RuntimeError('Run inside the AHR Go workspace')
        project = project.parent
    return project

def v4_import_exports(path, namespace):
    path = Path(path)
    book = nbformat.read(path, 4); nbformat.validate(book)
    count = 0
    for cell in book.cells:
        if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
            exec(compile(cell.source, str(path)+':ahr-export', 'exec'), namespace)
            count += 1
    if not count: raise RuntimeError('No tagged definitions in '+str(path))
    return count

def v4_load_upstream(project, namespace):
    project = Path(project)
    for number in (*range(7), 8, 9, 10, 11):
        paths = list((project/'notebooks').glob(f'{number:02d}_*.ipynb'))
        if len(paths) != 1: raise RuntimeError('Ambiguous original notebook '+str(number))
        v4_import_exports(paths[0], namespace)
    namespace['torch'].set_num_threads(1)
    return namespace

def v4_code_sha(path, exports_only=False):
    book = nbformat.read(path, 4); nbformat.validate(book)
    source = [cell.source for cell in book.cells if cell.cell_type == 'code' and
              (not exports_only or 'ahr-export' in cell.metadata.get('tags', []))]
    return hashlib.sha256(json.dumps(source, ensure_ascii=False).encode()).hexdigest()

def v4_source_identity(project, require_all=True):
    project = Path(project)
    paths = sorted((project/'notebooks').glob('*.ipynb'))
    for name in V4_NOTEBOOKS:
        path = project/V4_ROOT/name
        if path.exists(): paths.append(path)
        elif require_all: raise FileNotFoundError(path)
    return {str(path.relative_to(project)):v4_code_sha(path, True) for path in paths}

def v4_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()

def v4_read_json(path):
    return json.loads(Path(path).read_text())

def v4_report_path(project, name):
    if Path(name).name != name: raise ValueError('Simple report name required')
    directory = Path(project)/'reports/ahr_native_v4'; directory.mkdir(parents=True, exist_ok=True)
    return directory/name

def v4_seed(seed):
    import numpy as np, torch
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(seed); np.random.seed(seed % (2**32)); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def v4_atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    data = json.dumps(value, sort_keys=True, indent=2, allow_nan=False)+'\n'
    temporary = path.with_name('.'+path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with temporary.open('w') as stream:
            stream.write(data); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary, path)
        descriptor = os.open(str(path.parent), os.O_DIRECTORY)
        try: os.fsync(descriptor)
        finally: os.close(descriptor)
    finally:
        if temporary.exists(): temporary.unlink()
    return path


In [ ]:
PROJECT = v4_project()
v4_load_upstream(PROJECT, globals())
assert callable(play) and callable(terminal_go_targets) and callable(save_checkpoint)
print({'status':'PASS', 'workspace':str(PROJECT), 'torch':torch.__version__, 'training_device':V4_DEVICE,
       'rules':RULESET, 'source_notebooks':len(list((PROJECT/'notebooks').glob('*.ipynb')))})
